# Логистическая регрессия

In [109]:
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score

In [110]:
data = pd.read_csv('data-logistic.csv', header=None)
y = data.iloc[:, 0].values
X = data.iloc[:, 1:].values

In [111]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

In [112]:
def grad_desc(X, y, C=0, k=0.1, max_iter=10000, tol=1e-5, w0=None):
    if w0 is None:
        w = np.zeros(X.shape[1])
    else:
        w = w0.copy()
    for i in range(max_iter):
        grad = -np.mean((y[:, None] * X) * sigmoid(-y * np.dot(X, w))[:, None], axis=0)
        grad += C * w
        w_new = w - k * grad
        if np.linalg.norm(w_new - w) < tol:
            return w_new, i + 1
        w = w_new
    return w, max_iter

In [113]:
w_no_reg = grad_desc(X, y, C=0)[0]
w_reg = grad_desc(X, y, C=10)[0]

In [114]:
def predict(X, w):
    return sigmoid(np.dot(X, w))

In [115]:
y_no_reg = predict(X, w_no_reg)
y_reg = predict(X, w_reg)

In [116]:
auc_no_reg = roc_auc_score(y, y_no_reg)
auc_reg = roc_auc_score(y, y_reg)
print(round(auc_no_reg, 3), round(auc_reg, 3))

0.927 0.936


In [117]:
for k in [1, 0.5, 0.1, 0.05, 0.01, 0.001]:
    w, iters = grad_desc(X, y, C=0, k=k)
    print(k, iters)

1 32
0.5 60
0.1 244
0.05 431
0.01 1479
0.001 5126


При увеличении длины шага уменьшается количество итераций, но алгоритм может не сойтись, а при уменьшении числа итераций алгоритм сойдется, но потребуется большее число итераций

In [118]:
starts = [np.zeros(2), np.array([10, 10]), np.array([-10, -10]), np.array([100, -100])]
for w0 in starts:
    w, iters = grad_desc(X, y, C=0, k=0.1, w0=w0)
    print(w0, iters, np.round(w, 3))

[0. 0.] 244 [0.288 0.092]
[10 10] 526 [0.288 0.091]
[-10 -10] 317 [0.288 0.092]
[ 100 -100] 4405 [0.288 0.091]


C:\Users\karab\AppData\Local\Temp\ipykernel_20544\1702624272.py:2: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-z))


При разных стартах веса одинаковые (а значит и AUC одинаковый), но разное число итераций